# Stage 1 — search

Re-create this stage's script with Gemini's help. The cells below give you the spec, the seed, the gotchas, and an inline eval. The implementation itself is yours to write.


## 1. Setup


Every cell in this section is idempotent and safe to re-run. If you opened this notebook fresh (without running anything else in the same runtime), run them top-to-bottom.


### Clone the repo and `cd` into it


In [ ]:
# Bootstrap: clone the workshop repo into /content and cd into it.
# Idempotent — safe to re-run.
import os, subprocess, sys
REPO_DIR = "/content/ar-bic-2026-workshop"
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/jayprimer/ar-bic-2026-workshop.git", REPO_DIR],
        check=True,
    )
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())


### Install dependencies


Python (`openai`) and the Node CLI `@llamaindex/liteparse`. First run takes ~30s; re-runs are near-instant.


In [ ]:
# Install dependencies. Idempotent (pip skips already-installed; npm re-link is cheap).
# liteparse only matters for Stage 4 but installing it everywhere keeps each
# notebook self-contained, which is the whole point of re-running this cell.
!pip install -q -r requirements.txt
!npm install -g @llamaindex/liteparse 2>&1 | tail -3


### (no API key needed for this stage)


In [ ]:
# This stage doesn't call the OpenAI API.


## 2. Configure inputs


Edit the PubMed search inputs below. The variables in this cell drive your implementation directly — your script reads them by name, not from a file.


In [ ]:
# PubMed search inputs. Edit freely; canonical workshop values shown.
QUERY = (
    '("monoclonal antibody"[Title/Abstract] OR mAb[Title/Abstract]'
    ' OR "monoclonal antibodies"[Title/Abstract])'
    ' AND (pharmacokinetic*[Title/Abstract] OR toxicology[Title/Abstract]'
    ' OR toxicity[Title/Abstract] OR immunogenicity[Title/Abstract]'
    ' OR biodistribution[Title/Abstract])'
    ' AND (cynomolgus[Title/Abstract] OR "non-human primate"[Title/Abstract]'
    ' OR NHP[Title/Abstract] OR mouse[Title/Abstract]'
    ' OR rat[Title/Abstract] OR rodent[Title/Abstract])'
    ' AND ("2025"[Date - Publication] : "2026"[Date - Publication])'
)
N = 30                             # how many PMIDs to fetch (top N by date)
TOOL = "ar-bic-2026-workshop"
EMAIL = "workshop@example.org"

print(f"Query ({len(QUERY)} chars): {QUERY[:120]}...")
print(f"N={N}  TOOL={TOOL!r}  EMAIL={EMAIL!r}")


## 3. Spec — paste this into Gemini


Open the Gemini side panel in Colab (sparkles icon, top right) and paste the block below as your prompt. Then iterate.

```
Write Python that (top-to-bottom in the implementation cell, no
function-with-main wrapper needed):

1. Uses the QUERY, N, TOOL, EMAIL variables from the previous cell.
2. Calls NCBI E-utilities `esearch.fcgi` (db=pubmed, retmax=N,
   retmode=json, sort=date) and assigns the PMID list to a variable
   named `pmids`.
3. Calls NCBI E-utilities `efetch.fcgi` (db=pubmed, retmode=xml) for
   those PMIDs and parses each record into a dict with these keys:
   pmid, title, abstract, authors (list), first_author, journal,
   year (int), pub_types (list). Assign to a variable named
   `records`.
4. Re-sorts `records` to match the esearch `pmids` order (efetch
   ordering is not guaranteed).
5. Asserts: `len(records) == len(pmids)`, and every record has a
   non-empty `pmid` and `title`.

Use only the standard library (urllib, json, xml.etree.ElementTree).
Do NOT use Biopython or `requests`. Do NOT save to disk — the next
cell handles persistence.
```


## 4. Gotchas Gemini probably won't know


Copy any that apply into Gemini if it goes off-track:

- **Abstracts are nested XML.** Use `el.itertext()` joined together,
  NOT `el.text`, when reading `<ArticleTitle>` and `<AbstractText>`.
  `.text` silently truncates at the first inline child (`<i>`,
  `<sub>`, `<sup>`).
- **Abstract can be multi-part labeled.** Find all
  `.//Abstract/AbstractText`, read each `Label` attribute, and join
  them as `"BACKGROUND: ... METHODS: ..."`.
- **efetch can reorder.** Re-sort `records` to match the esearch
  `pmids` order before saving.
- **Identify yourself to NCBI.** Pass `tool` and `email` URL params
  (anonymous clients are throttled hard).


## 5. Seed — a few lines to anchor Gemini


In [ ]:
import json, os, urllib.parse, urllib.request
import xml.etree.ElementTree as ET
HEADERS = {"User-Agent": "ar-bic-2026/0.1"}

# Implementation goes in the next cell.
# Produce two variables for the inspect cell to consume:
#   pmids   : list[str]
#   records : list[dict]


## 6. Your implementation


Drive Gemini to fill this in. Iterate until the inspect cell below shows reasonable output and the eval cell passes.


In [ ]:
# TODO: implement Stage 1 here.
# Read the spec above. Use the seed cell's imports.
# When done, run the inspect + eval cells next.


## 7. Inspect output


In [ ]:
# Save your results to disk for downstream stages + eval, then display.
import json, os
os.makedirs("stage_01/data", exist_ok=True)
with open("stage_01/data/pmids.json", "w") as f:
    json.dump({"query": QUERY, "n_requested": N,
               "pmids": pmids, "records": records}, f, indent=2)
print(f"Saved {len(records)} records → stage_01/data/pmids.json\n")

print("First 3 records:")
for r in records[:3]:
    print(f"  PMID {r['pmid']}: {r['title'][:78]}")
    print(f"    {r.get('first_author','?')} ({r.get('year','?')}) · "
          f"{(r.get('journal') or '')[:50]}")
    print(f"    Abstract: {(r.get('abstract') or '')[:140]}…\n")


## 8. Run eval


Inline eval — same checks as `eval/eval_01_script.py`, but the code is right here so you can see what it's measuring. Writes `stage_01/eval/eval_script.json` + `score.json`.


In [ ]:
# Eval — same checks as eval/eval_01_script.py, inlined so you can see
# what's being measured. Reads stage_01/data/pmids.json from disk.
import datetime, json, os, re

XML_TAG_LEAK = re.compile(r"</?[a-zA-Z]")
REQUIRED_FIELDS = {"pmid", "title", "abstract", "authors", "first_author",
                   "journal", "year", "pub_types"}
THIS_YEAR = datetime.date.today().year

os.makedirs("stage_01/eval", exist_ok=True)
with open("stage_01/data/pmids.json") as f:
    doc = json.load(f)
pmids   = doc["pmids"]
records = doc.get("records") or []

# ---- PMID-list shape ----
checks = {
    "pmids_non_empty":   bool(pmids),
    "all_numeric":       all(p.isdigit() for p in pmids),
    "no_duplicates":     len(pmids) == len(set(pmids)),
    "respects_cap":      len(pmids) <= doc.get("n_requested", len(pmids)),
    "records_match_pmids": [r["pmid"] for r in records] == pmids,
}

# ---- per-record completeness ----
n = len(records)
counts = dict(fields=0, title=0, abs_long=0, abs_clean=0,
              authors=0, journal=0, year=0, pub_types=0)
issues = []
for r in records:
    rec_issues = []
    if set(r.keys()) >= REQUIRED_FIELDS: counts["fields"] += 1
    else: rec_issues.append(f"missing keys: {REQUIRED_FIELDS - set(r.keys())}")
    if r.get("title") and len(r["title"]) > 10: counts["title"] += 1
    else: rec_issues.append("title missing or < 10 chars")
    ab = r.get("abstract") or ""
    if len(ab) > 200: counts["abs_long"] += 1
    if not XML_TAG_LEAK.search(ab): counts["abs_clean"] += 1
    else: rec_issues.append("abstract contains XML-tag-shaped leakage")
    if r.get("authors"): counts["authors"] += 1
    if r.get("journal"): counts["journal"] += 1
    yr = r.get("year")
    if isinstance(yr, int) and 1990 <= yr <= THIS_YEAR + 1: counts["year"] += 1
    else: rec_issues.append(f"year out of range: {yr!r}")
    if r.get("pub_types"): counts["pub_types"] += 1
    else: rec_issues.append("pub_types empty")
    if rec_issues: issues.append({"pmid": r.get("pmid"), "issues": rec_issues})

checks.update({
    "all_records_have_required_fields": counts["fields"] == n,
    "all_titles_substantive":           counts["title"] == n,
    "no_xml_tag_leakage_in_abstracts":  counts["abs_clean"] == n,
    "most_abstracts_full_length":       (counts["abs_long"]/n) >= 0.6 if n else True,
    "most_records_have_authors":        (counts["authors"]/n) >= 0.9 if n else True,
    "all_journals_named":               counts["journal"] == n,
    "all_years_in_range":               counts["year"] == n,
    "all_records_have_pub_types":       counts["pub_types"] == n,
})

print("Script checks:")
for k, v in checks.items():
    print(f"  {'OK  ' if v else 'FAIL'}  {k}")
if issues:
    print(f"\nPer-record issues ({len(issues)}):")
    for it in issues[:5]:
        print(f"  {it['pmid']}: {'; '.join(it['issues'])}")
    if len(issues) > 5:
        print(f"  …and {len(issues)-5} more")

with open("stage_01/eval/eval_script.json", "w") as f:
    json.dump({"script": checks, "per_record_issues": issues}, f, indent=2)
n_pass = sum(1 for v in checks.values() if v); n_total = len(checks)
score_path = "stage_01/eval/score.json"
score = json.load(open(score_path)) if os.path.exists(score_path) else {}
score["script"] = {"passed": n_pass, "total": n_total,
                   "percent": round(100*n_pass/n_total, 1)}
with open(score_path, "w") as f: json.dump(score, f, indent=2)
print(f"\nScore: {n_pass}/{n_total} ({score['script']['percent']}%)")


## 9. Stuck? Skip this stage


Copy the reference run's Stage 1 output into place so the next stage's notebook can still run. Use this sparingly — the point of the workshop is to *re-create* each stage.


In [ ]:
import os, shutil
os.makedirs("stage_01/data", exist_ok=True)
shutil.copy("reference_outputs/stage_01/data/pmids.json",
            "stage_01/data/pmids.json")
print("copied reference Stage 1 output")
